# Load depedencies

## Install depedencies

In [ ]:
%%capture
%pip install -U langchain langchainhub langchain_community langchain-huggingface
%pip install -U faiss-gpu transformers accelerate

## Import depedencies

In [2]:
import torch

from IPython.display import clear_output
from langchain.document_loaders import HuggingFaceDatasetLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain.vectorstores import FAISS
from transformers import AutoTokenizer, AutoModelForCausalLM,pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain.chains import RetrievalQA
from langchain import hub

# Load Dataset

In [ ]:
dataset_path = "tatsu-lab/alpaca"
dataset_loader = HuggingFaceDatasetLoader(dataset_path, "output")

## Select te first 1000 entries
alpaca_loader = dataset_loader.load()
alpaca_loader = alpaca_loader[:1000]
alpaca_loader[:2]

In [ ]:
dataset_path = "shaowenchen/wiki_zh"
dataset_loader = HuggingFaceDatasetLoader(dataset_path, "text")

#dataset_path = "llm-wizard/alpaca-gpt4-data-zh"
#dataset_loader = HuggingFaceDatasetLoader(dataset_path, "output")

wiki_zh_loader = dataset_loader.load()
## Select te first 1000 entries
wiki_zh_loader = wiki_zh_loader[:20000]

# wiki_zh_loader[:2]

# Load Embedding model

In [ ]:
# Define the path to the pre-trained model you want to use
# modelPath = "sentence-transformers/all-MiniLM-L12-v2"
modelPath = "BAAI/bge-large-zh-v1.5"

# Create a dictionary with model configuration options, specifying to use the GPU for computations
model_kwargs = {'device':'cuda'}

# Create a dictionary with encoding options, specifically setting 'normalize_embeddings' to False
encode_kwargs = {'normalize_embeddings': False}

# Initialize an instance of HuggingFaceEmbeddings with the specified parameters
embeddings = HuggingFaceEmbeddings(
    model_name=modelPath,     
    model_kwargs=model_kwargs, 
    encode_kwargs=encode_kwargs
)

In [7]:
text = "你好啊月亮"
query_result = embeddings.embed_query(text)
query_result[:3]

[0.07670598477125168, -0.004631328862160444, -0.04397747293114662]

## Create a VectorDB

In [ ]:
vector_db = FAISS.from_documents(wiki_zh_loader, embeddings)
vector_db.save_local("/kaggle/working/faiss_doctor_index")

In [ ]:
# help(vector_db)

In [ ]:
question = "三原色是什么？"
searchDocs = vector_db.similarity_search(question)
x = searchDocs[0]
x

In [ ]:
s = x.page_content
print(s.encode().decode("unicode_escape"))

# Load Model

In [ ]:
base_model = "/kaggle/input/qwen2.5/transformers/7b-instruct/1"

tokenizer = AutoTokenizer.from_pretrained(base_model)

model = AutoModelForCausalLM.from_pretrained(
        base_model,
        return_dict=True,
        low_cpu_mem_usage=True,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
)

## Create Pipeline

In [ ]:
pipe = pipeline(
    "text-generation", 
    model=model, 
    tokenizer=tokenizer,
    max_new_tokens=1024
)

llm = HuggingFacePipeline(pipeline=pipe)

# Create chain

In [ ]:
retriever = vector_db.as_retriever()
rag_prompt = hub.pull("rlm/rag-prompt")

In [ ]:
qa_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

## Testing

In [ ]:
question_1 = "三原色是什么"
result_1 = qa_chain.invoke(question_1)

print(result_1.split("Answer: ")[1])

In [ ]:
question_2 = "生成一个人旅行需要的东西"
result_2 = qa_chain.invoke(question_2)
print(result_2.split("Answer: ")[1])

In [ ]:
question_3 = "为什么资本主义有生产过剩危机"
result_3 = qa_chain.invoke(question_3)

print(result_3.split("Answer: ")[1])